# WM 2026 — Pro-Spiel-Prognose (Ebene A)

Interaktive Sicht auf die 1/X/2-Wahrscheinlichkeiten und das wahrscheinlichste
Resultat pro Spiel. Gleiche Engine wie das tägliche Automatik-Skript
(`src/live_forecast.py`), nur dass du hier flexibel filtern und gruppieren kannst.

**Was hier drin ist:**
1. Modell laden (Vorturnier-Prognose, kein Live-Abruf)
2. Alle noch offenen Spiele anzeigen
3. Nach Team filtern (Spielplan eines bestimmten Teams)
4. Nach Datum filtern (z.B. Spiele dieser Woche)
5. Die spannendsten Spiele (möglichst ausgeglichen)

## 1. Setup & Modell laden

Pfad wird robust aufgelöst (egal von welchem cwd der Kernel startet).
`return_context=True` gibt zusätzlich die Engine zurück — die brauchen wir für
die Pro-Spiel-Prognose.

In [1]:
import sys
from pathlib import Path

# Robust: src/wm_model.py finden
_found = None
for parent in [Path.cwd(), *Path.cwd().parents]:
    for cand in (parent / "src", parent / "wm2026" / "src"):
        if (cand / "wm_model.py").exists():
            _found = cand; break
    if _found: break
if _found is None:
    raise FileNotFoundError("wm_model.py nicht gefunden — Notebook aus dem wm2026-Projekt heraus öffnen.")
sys.path.insert(0, str(_found))
import wm_model as wm

result, n, ctx = wm.build_forecast(
    local=str(_found.parent / "data" / "results.csv"),
    live_results=None,                                          # kein Live-Abruf — interaktiv
    fit_cache=str(_found.parent / "output" / "goal_coefs.json"),
    return_context=True,
)
matches = wm.predict_remaining_matches(ctx["df_all"], ctx["engine"], ctx["known"])
print(f"Geladen: {len(matches)} offene Spiele · {n} Live-Resultate berücksichtigt")

Geladen: 72 offene Spiele · 0 Live-Resultate berücksichtigt


## 2. Alle offenen Spiele

`⌂` markiert ein Heimspiel eines Gastgebers (Mexico/USA/Canada) — dort fließt
der Heimvorteil-Bonus mit ein.

In [2]:
_fmt = {"p_home": "{:.0%}", "p_draw": "{:.0%}", "p_away": "{:.0%}",
        "p_most_likely": "{:.1%}"}

def show(df, rows=None):
    '''Hübsche Anzeige mit ⌂-Marker für Gastgeber-Heimspiele.'''
    d = df.copy()
    d["home_team"] = d.apply(lambda r: f"{r['home_team']} ⌂" if r["home_advantage"] else r["home_team"], axis=1)
    d = d.drop(columns=["home_advantage"])
    return (d.head(rows) if rows else d).style.format(_fmt)

show(matches, 15)

,date,home_team,away_team,p_home,p_draw,p_away,most_likely,p_most_likely
0,2026-06-11,Mexico ⌂,South Africa,81%,13%,7%,2:0,12.0%
1,2026-06-11,South Korea,Czech Republic,47%,27%,27%,1:0,12.7%
2,2026-06-12,Canada ⌂,Bosnia and Herzegovina,75%,15%,9%,2:0,11.9%
3,2026-06-12,United States ⌂,Paraguay,36%,25%,38%,1:1,12.0%
4,2026-06-13,Qatar,Switzerland,7%,15%,77%,0:2,14.5%
5,2026-06-13,Brazil,Morocco,44%,27%,29%,1:1,12.8%
6,2026-06-13,Haiti,Scotland,24%,26%,50%,0:1,13.0%
7,2026-06-13,Australia,Turkey,31%,28%,41%,1:1,13.0%
8,2026-06-14,Germany,Curaçao,78%,15%,7%,2:0,14.5%
9,2026-06-14,Ivory Coast,Ecuador,16%,23%,61%,0:1,13.6%


## 3. Spielplan eines Teams

Spalte `TEAM` setzen und die Cell ausführen — zeigt alle Spiele dieses Teams
(Heim oder Auswärts) mit Prognose.

In [3]:
TEAM = "Switzerland"

mask = (matches["home_team"] == TEAM) | (matches["away_team"] == TEAM)
team_matches = matches[mask].sort_values("date")
print(f"{TEAM}: {len(team_matches)} offene Spiele")
show(team_matches)

Switzerland: 3 offene Spiele


,date,home_team,away_team,p_home,p_draw,p_away,most_likely,p_most_likely
4,2026-06-13,Qatar,Switzerland,7%,15%,77%,0:2,14.5%
26,2026-06-18,Switzerland,Bosnia and Herzegovina,70%,19%,11%,2:0,13.7%
50,2026-06-24,Canada ⌂,Switzerland,40%,25%,35%,1:1,12.0%


## 4. Spiele in den nächsten Tagen

Zeigt alle Spiele in einem Datumsfenster — praktisch, um die Spieltage der
kommenden Woche auf einen Blick zu haben.

In [4]:
VON, BIS = "2026-06-11", "2026-06-14"     # erste Spieltage anpassen

window = matches[(matches["date"] >= VON) & (matches["date"] <= BIS)].sort_values("date")
print(f"{VON} bis {BIS}: {len(window)} Spiele")
show(window)

2026-06-11 bis 2026-06-14: 12 Spiele


,date,home_team,away_team,p_home,p_draw,p_away,most_likely,p_most_likely
0,2026-06-11,Mexico ⌂,South Africa,81%,13%,7%,2:0,12.0%
1,2026-06-11,South Korea,Czech Republic,47%,27%,27%,1:0,12.7%
2,2026-06-12,Canada ⌂,Bosnia and Herzegovina,75%,15%,9%,2:0,11.9%
3,2026-06-12,United States ⌂,Paraguay,36%,25%,38%,1:1,12.0%
4,2026-06-13,Qatar,Switzerland,7%,15%,77%,0:2,14.5%
5,2026-06-13,Brazil,Morocco,44%,27%,29%,1:1,12.8%
6,2026-06-13,Haiti,Scotland,24%,26%,50%,0:1,13.0%
7,2026-06-13,Australia,Turkey,31%,28%,41%,1:1,13.0%
8,2026-06-14,Germany,Curaçao,78%,15%,7%,2:0,14.5%
9,2026-06-14,Ivory Coast,Ecuador,16%,23%,61%,0:1,13.6%


## 5. Die spannendsten Spiele

Ein „spannendes" Spiel ist eines, bei dem das Modell sich am wenigsten sicher
ist — d.h. die drei Wahrscheinlichkeiten 1/X/2 liegen nah beieinander. Wir
messen das mit der **Entropie** der Wahrscheinlichkeitsverteilung: maximale
Entropie = 33%/33%/33% (totale Unsicherheit), niedrige Entropie = klarer Favorit.

In [5]:
import numpy as np

def _entropy(row):
    p = np.array([row["p_home"], row["p_draw"], row["p_away"]])
    p = p[p > 0]
    return -(p * np.log(p)).sum()

exciting = matches.assign(spannung=matches.apply(_entropy, axis=1))
exciting = exciting.nlargest(10, "spannung").drop(columns=["spannung"])
print("Top 10 spannendste offene Spiele:")
show(exciting)

Top 10 spannendste offene Spiele:


,date,home_team,away_team,p_home,p_draw,p_away,most_likely,p_most_likely
55,2026-06-25,Paraguay,Australia,37%,28%,35%,1:1,13.1%
41,2026-06-22,Norway,Senegal,38%,28%,34%,1:1,13.1%
57,2026-06-25,Ecuador,Germany,39%,28%,34%,1:1,13.1%
62,2026-06-26,Cape Verde,Saudi Arabia,34%,28%,39%,1:1,13.1%
11,2026-06-14,Sweden,Tunisia,39%,28%,34%,1:1,13.1%
68,2026-06-27,Colombia,Portugal,39%,28%,33%,1:1,13.1%
10,2026-06-14,Netherlands,Japan,39%,28%,33%,1:1,13.1%
66,2026-06-27,Algeria,Austria,33%,28%,40%,1:1,13.1%
31,2026-06-19,Turkey,Paraguay,40%,28%,32%,1:1,13.0%
7,2026-06-13,Australia,Turkey,31%,28%,41%,1:1,13.0%


## Weitere Ideen

- **Schiefster Favorit:** `matches.nlargest(5, "p_home")` bzw. `nlargest(5, "p_away")`
  → die deutlichsten Vorhersagen.
- **Höchstes erwartetes Tor-Spektakel:** `most_likely`-Spalte parsen,
  Summe der Tore sortieren.
- **Verlauf einzelner Spiele über die Zeit:** sobald die Automatik mehrere
  Tage gelaufen ist, liefert `output/history.csv` die Titel-Verläufe der
  Teams. Pro-Spiel-Verlauf bräuchte ein analoges `matches_history.csv` —
  Erweiterung wäre eine Zeile im Skript.